---
title: "Cost-per-click surge attribution"
author: "Andratx Bellmunt"
abstract: >
  Adapted from a real business scenario. We use mix-rate decomposition to asses the contribution of each individual product to a global surge on cost per click. In order to get bootstrap confidence intervals on each product contribution we rely on Monte Carlo methods. 
format:
  html:
    code-fold: true
    self-contained: true
    include-after-body: _tracker.html
jupyter: python3
number-sections: false
---

# Initialization

## Imports and settings

In [ ]:
import pandas as pd
import plotly.graph_objects as go

In [ ]:
pd.options.display.float_format = "{:,.2f}".format

## Auxiliary functions

In [ ]:
def add_derived_columns(df: pd.DataFrame) -> pd.DataFrame:
    df["cpc_before"] = df["cost_before"] / df["clicks_before"]
    df["cpc_after"] = df["cost_after"] / df["clicks_after"]
    df["cpc_diff"] = df["cpc_after"] - df["cpc_before"]

    return df

def aggregate_data(df: pd.DataFrame) -> pd.DataFrame:
    df_agg = pd.DataFrame(df.sum()).T
    df_agg["clicks_before"] = df_agg["clicks_before"].astype(int)
    df_agg["clicks_after"] = df_agg["clicks_after"].astype(int)

    return df_agg

def compute_rate_mix_effects(df: pd.DataFrame) -> pd.DataFrame:
    clicks_total_after = df["clicks_after"].sum()
    clicks_total_before = df["clicks_before"].sum()
    df["rate_effect"] = df["clicks_after"] / clicks_total_after * df["cpc_diff"]
    df["mix_effect"] = (df["clicks_after"] / clicks_total_after - df["clicks_before"] / clicks_total_before) * df["cpc_before"]
    df["total_effect"] = df["rate_effect"] + df["mix_effect"]

    return df


# Understanding the business problem

## The problem

In [ ]:
df_toy = pd.DataFrame(
    columns=["id", "cost_before", "clicks_before", "cost_after", "clicks_after"],
    data=[
        ["A", 10000.00, 200, 1500.00, 25],
        ["B",   800.00,  40,  900.00, 40]
    ]
)

df_toy = add_derived_columns(df_toy)
df_toy = compute_rate_mix_effects(df_toy)

df_agg = aggregate_data(df_toy)
df_agg = add_derived_columns(df_agg)

print("Toy example:")
display(df_toy)
print("Aggregated data:")
display(df_agg)

Even in a simple example with only two products we can see how the phenomenon of Simpson's paradox arises:

- Both products individual CPC goes up: +$10 for product A and +$2.50 for product B

- However the aggregated CPC goes down: -$8.08

This example actually illustrates a very common business situation:

- Product A runs under some algorithm that optimizes RPS

- The algorithm discards sales with low return and the final effect is that both revenue and sales decrease, albeit in a way that RPS increases (in other words: the percentual decrease in revenue is less than the percentual decrease in sales)

- When aggregating the data with other products this has a harming impact on the global RPS

In a real case scenario where we have ~4,000 different products instead of just a couple, these interactions become much more complex.

## The initial solution and why it doesn't work

In order to navigate the paradox we borrow a tool from game theory: [Shapley values](https://en.wikipedia.org/wiki/Shapley_value).

- Shapley values measure the contribution of each individual player to a common goal

- To us each product is a player and the common goal is the aggregated RPS. We want to measure how much each individual product contributes to it.

- One property of Shapley values is that individual contributions always add up to the final global result. In our example above, the sum of the Shapley value of A and the Shapley value of B must be -17.95.

*Note:* It is not our intend to provide a detailed account on how Shapley values are computed. In the present section we ask the reader to trust us, while in future more technical sections we assume the reader has enough familiarity with the concept.

In our example we have v(A) = $10.00, v(B) = $2.50, v(A,B) = -$8.08. Then:

- s(A) = 1/2 [v(A,B) + v(A) - v(B)] = -0.29

- s(B) = 1/2 [v(A,B) + v(B) - v(A)] = -7.79

## Real data

In [ ]:
df = pd.read_csv("../assets/cpc_data.csv")

In [ ]:
df = compute_rate_mix_effects(add_derived_columns(df))

In [ ]:
df_agg = compute_rate_mix_effects(add_derived_columns(aggregate_data(df)))

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(x=df["cpc_before"], y=df["cpc_after"], mode="markers", marker_size=2.5))
fig.add_trace(go.Scatter(x=[0,80], y=[0,80], mode="lines", marker_color="red", marker_type="dash"))

fig.update_layout(
    title="CPC comparison (before vs after)"
    width=600,
    height=600,
    xaxis_title="CPC before",
    yaxis_title="CPC_after",
    yaxis_scaleanchor="x",
    yaxis_scaleratio=1,
)

fig.show()